# CNN inverse training on Colab T4

Single in-memory pipeline (gen + train, no disk roundtrip — Colab session storage is slow).

Iteration loop: edit `notebooks/cnn_pipeline.py` locally → `git push` → run the `git pull` cell → re-run the pipeline cell. ~5-10 min per iteration.

In [ ]:
# 1) Clone repo, install as package (all deps from pyproject.toml), verify GPU
import os, sys, subprocess

REPO_DIR = "/content/water_v2"
REPO_URL = "https://github.com/alexhrubin/water_v2.git"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-b", "python-rewrite", REPO_URL, REPO_DIR], check=True)
    print(f"Cloned to {REPO_DIR}")
else:
    print(f"{REPO_DIR} already exists; run the `git pull` cell below to update.")

os.chdir(REPO_DIR)
subprocess.run(["pip", "install", "-q", "-e", REPO_DIR], check=True)

import jax
print(f"JAX backend: {jax.default_backend()}  devices: {jax.devices()}")
from wavetank import Tank, build_propagator   # noqa: F401
print("wavetank imports OK")

In [ ]:
# Pull latest after editing scripts locally and `git push`-ing
!cd /content/water_v2 && git pull

In [ ]:
# 2) Generate + train in one pass (in-memory; ~30s gen + ~5-10 min training on T4)
# Saves model_cnn.eqx + history_cnn.json + metadata.json. No 17 GB npz roundtrip.
!python -u notebooks/cnn_pipeline.py

In [ ]:
# 3) OOD eval
!python -u notebooks/eval_cnn_inverse.py

In [ ]:
from IPython.display import Image, display
display(Image("data/naive_inverse/eval_cnn_ood.png"))